# MetroPT temporal tabular experiment

This notebook runs the fixed current-window versus temporal-context comparison across 6/12/24/48-hour horizons. It uses rolling-origin May, June and July folds; model and threshold selection use May and June only. Keep a standard CPU runtime and select **Runtime → Run all**. Expected duration is approximately 10–30 minutes.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile

repo = Path('/content/metropt3-predictive-maintenance')
branch = 'investigation/temporal-validation'
if repo.exists():
    subprocess.run(['git', 'fetch', 'origin', branch], cwd=repo, check=True)
    subprocess.run(['git', 'checkout', branch], cwd=repo, check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{branch}'], cwd=repo, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', branch, '--single-branch', 'https://github.com/SahilBh01r1769/metropt3-predictive-maintenance.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-experiment.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Running revision:', revision)

## Run the complete tabular matrix

This downloads the audited UCI source, rebuilds validated one-hour windows and causal temporal features, then fits Dummy, Logistic Regression, Random Forest and XGBoost models. Do not alter the configuration after viewing results.

In [ ]:
from metropt3.audit import file_sha256
from metropt3.config import RAW_FILENAME

subprocess.run([sys.executable, 'scripts/download_data.py'], check=True)
csv_path = Path('data') / RAW_FILENAME
expected_hash = 'db30ccb4ea402e3c8bf2c99db06e288d4f2a772f6928f9dbe26a920d69793e24'
assert file_sha256(csv_path) == expected_hash, 'Dataset does not match the audited source'
output = Path('/content/metropt_tabular_output')
if output.exists():
    shutil.rmtree(output)
subprocess.run([sys.executable, 'scripts/run_tabular_experiment.py', '--csv', str(csv_path), '--output', str(output)], check=True)

## Validate and download

The final cell refuses to package the run unless every requested comparison artifact and the selected model are present. It downloads `metropt_tabular_evidence.zip` through the browser; the ZIP is not placed in Google Drive. Attach that downloaded ZIP in the chat.

In [ ]:
required = [
    'horizon_comparison.csv', 'model_comparison.csv', 'event_metrics.csv',
    'threshold_analysis.csv', 'fold_results.csv', 'feature_importance.csv',
    'selected_thresholds.csv', 'predictions.csv.gz', 'experiment_config.json',
    'execution_manifest.json', 'run_summary.json', 'validation.json',
    'best_tabular_model.joblib',
]
missing = [name for name in required if not (output / name).exists()]
assert not missing, f'Missing experiment evidence: {missing}'
bundle = Path('/content/metropt_tabular_evidence.zip')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in output.rglob('*'):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(output))
print('Evidence bundle:', bundle, f'({bundle.stat().st_size / 1_000_000:.1f} MB)')
from google.colab import files
files.download(str(bundle))